# CoLLM-QFormer — Colab

Candidate-aware Q-Former CIE for CoLLM. Run the cells in order.

`RUNSHEET_COLAB.md` is the authoritative doc — it explains *why* for each step, lists the
measured numbers, the search order, the ablation commands and the failure modes. This
notebook is the executable shell around it.

**Requires an A100 (40 GB) runtime.** L4 works with reduced batches; T4 will not fit.

## 0. Runtime check

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

## 1. Environment

Colab runs **Python 3.12**, and CoLLM's own `requirements.txt` cannot be installed there:
`transformers==4.28.0` needs `tokenizers<0.14`, which has no cp312 wheel (pip tries a Rust
source build and fails); `scikit-learn==1.2.2` compiles from source; `decord` has no wheel
for Python ≥ 3.11 at all.

The pins below have both bounds for a reason — see runsheet §1.

| Package | Pin | Why |
|---|---|---|
| `transformers` | 4.36.2 | ≥4.34 for a cp312 `tokenizers`; <5 because CoLLM vendors `modeling_llama.py` against the 4.x API |
| `peft` | 0.9.0 | 0.10.0 removed `prepare_model_for_int8_training`, imported by name |
| `scikit-learn`, `numpy` | unpinned | the NaN-UAUC problem is fixed in code, not by a pin |
| `decord` | not installed | stubbed by `qformerrec/compat.py` |

### 1.1 Mount Drive and clone both repos

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

%cd /content
!git clone -q https://github.com/zyang1580/CoLLM.git         # the baseline
!git clone -q https://github.com/htainvn/QFormerRec.git      # this work

import os
os.environ["COLLM_ROOT"] = "/content/CoLLM"
%cd /content/QFormerRec
!mkdir -p /content/data /content/ckpt /content/logs
!git log --oneline -1        # record the commit you ran
!ls

No install needed for the package itself — `train_qformer.py` puts both trees on
`sys.path`. Iterating on the code? **Pull, don't re-clone** (a re-clone wipes the config
edits from §6/§7):

```
%cd /content/QFormerRec && git pull -q && git log --oneline -1
```

Private repo? The anonymous HTTPS clone 404s (Colab has no SSH key); use
`getpass()` + a fine-grained PAT — see runsheet §1.1.

### 1.2 Install the pinned stack

In [ ]:
!pip -q install -r requirements.txt
# equivalently:
# !pip -q install "transformers==4.36.2" "peft==0.9.0" omegaconf webdataset timm \
#     iopath sentencepiece opencv-python-headless scikit-learn pandas scipy

Expect two harmless things:

1. **Conflict warnings** naming `gradio` / the preinstalled `transformers` 5.x and
   `huggingface_hub` 1.x. Colab ships transformers 5.x; pinning 4.36.2 downgrades the hub
   below 1.0. Nothing here uses gradio.
2. **Restart the runtime now** (`Runtime → Restart session`), then resume at §1.3 —
   otherwise you may keep running the old transformers already in memory.

### 1.3 Verify the environment

In [ ]:
%cd /content/QFormerRec
import os; os.environ["COLLM_ROOT"] = "/content/CoLLM"   # re-set after a restart
!python -c "from qformerrec.compat import check_environment as c; import sys; sys.exit(1 if c() else 0)" \
    && echo "environment OK"

## 2. Data

In [ ]:
%cd /content/data
!unzip -o -q /content/CoLLM/collm-datasets/ml-1m.zip
!unzip -o -q /content/CoLLM/collm-datasets/amazon_book.zip
!ls ml-1m book

import pandas as pd
for s in ["train", "valid_small", "test"]:
    d = pd.read_pickle(f"/content/data/ml-1m/{s}_ood2.pkl")
    print(f"{s:12s} rows={len(d):6d} users={d.uid.nunique():4d} pos={d.label.mean():.3f}")

In [ ]:
# ML-1M genres (optional; needed for --genre_source metadata)
%cd /content/data
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O ml1m_raw.zip
!unzip -o -q ml1m_raw.zip 'ml-1m/movies.dat' -d raw
!ls -la raw/ml-1m/movies.dat
%cd /content/QFormerRec

## 3. Stage 0 — pretrain MF

Gate: ML-1M test AUC ~0.6482 / UAUC ~0.6361. If it misses, stop.

In [ ]:
!python scripts/pretrain_mf.py --data_dir /content/data/ml-1m/ \
    --out /content/ckpt/mf_ml1m_d256.pth 2>&1 | tail -20

In [ ]:
!python scripts/pretrain_mf.py --data_dir /content/data/book/ \
    --out /content/ckpt/mf_amazon_d256.pth 2>&1 | tail -20

## 4. Stage 0b — build the train-only fitted objects

This artifact holds the KNN neighbour graph, the KMeans centroids, the genre prototype
means and `item_in_train`. It does **not** hold the history slots: those are per-row and
point-in-time, read from each sample's own `his` column (the same list CoLLM renders into
`<ItemTitleList>`), with no precomputation.

The fitted objects stay train-split-only — fitting them across valid/test would use other
users' futures, which is real leakage. Amazon-Book has no category field, so it uses
KMeans pseudo-genres (use a high-RAM runtime — its train pickle expands to several GB).

In [ ]:
!python scripts/build_memory.py \
    --data_dir /content/data/ml-1m/ --mf_ckpt /content/ckpt/mf_ml1m_d256.pth \
    --out /content/ckpt/memory_index_ml1m.pkl --dataset ml1m \
    --genre_source metadata --movies_dat /content/data/raw/ml-1m/movies.dat

In [ ]:
!python scripts/build_memory.py \
    --data_dir /content/data/book/ --mf_ckpt /content/ckpt/mf_amazon_d256.pth \
    --out /content/ckpt/memory_index_amazon.pkl --dataset amazon_book \
    --genre_source item_kmeans --n_user_clusters 256

## 5. Self-checks — run these before burning GPU hours

Three gates, all CPU, no Vicuna needed. The first one is mandatory whenever the data or
`k_hist` changes: it proves the history is point-in-time and, crucially, that the per-row
path actually took effect.

In [ ]:
# Point-in-time history gate. Asserts: split ordering, `his` is positives-only,
# per-row causality (ties allowed, zero future items), index 0 is leading padding only.
# Then prints the provenance of the k_hist items actually used -- for ML-1M test at
# k_hist=10 expect ~9% train / 14% valid / 77% test. Mostly `train` => not in effect.
!python scripts/check_pit_history.py --data_dir /content/data/ml-1m/ --k_hist 10 20 50

In [ ]:
!python scripts/check_pit_history.py --data_dir /content/data/book/ --k_hist 10 50

In [ ]:
!python scripts/smoke_test.py --memory_index /content/ckpt/memory_index_ml1m.pkl 2>&1 | tail -25

In [ ]:
!python scripts/integration_test.py --data_dir /content/data/ml-1m/ \
    --mf_ckpt /content/ckpt/mf_ml1m_d256.pth \
    --memory_index /content/ckpt/memory_index_ml1m.pkl \
    --work_dir /tmp/itest 2>&1 | grep -E "^\[|passed|FAILED" | tail -40

## 6. Point the configs at Vicuna-7B

CoLLM's published numbers are on **Vicuna-7B v0**. If you use v1.x you must re-run the
CoLLM-MF and TALLRec baselines yourself — the reference table no longer applies.

### Vicuna-7B v0

`lmsys/vicuna-7b-v0` **does not exist as merged weights** — only the delta, which cannot be
loaded directly. Pick one (runsheet §5.0 has the trade-offs):

- **(a) official delta** — matches the paper; ~27 GB plus a merge step
- **(b) community merged mirror** — fastest, unverified provenance
- **(c) `lmsys/vicuna-7b-v1.5`** — no delta step, but then you must re-run the CoLLM-MF and
  TALLRec baselines yourself, since the reference table is v0-only

`snapshot_download` is used instead of `huggingface-cli` deliberately: that command was
renamed to `hf` in `huggingface_hub` 0.34 and removed in 1.0, so which one exists depends on
whether you run this before or after §1.2. Download to `/content`, **not** Drive — a
Drive-mounted 13.5 GB load costs minutes of FUSE overhead every session.

In [ ]:
# (b), the fastest way to unblock. For (a) or (c) see runsheet 5.0.
from huggingface_hub import snapshot_download
snapshot_download("ZzZZCHS/vicuna-7b-v0", local_dir="/content/vicuna-7b-v0")

In [ ]:
VICUNA = "/content/vicuna-7b-v0"     # whatever you produced above
!sed -i "s#/content/vicuna-7b-v0#$VICUNA#" train_configs/*.yaml
!ls $VICUNA
!grep -h llama_model train_configs/stage2_qformer_ml1m.yaml

## 7. Stage 1 — LoRA warmup (once per dataset, then cached forever)

Unchanged CoLLM stage 1: TALLRec prompt *with* history titles, LoRA only. Independent of
every Q-Former hyper-parameter, so all ablations reuse it. Batch reduced to 16 for a
40 GB A100.

In [ ]:
!sed -i 's#batch_size_train: 32#batch_size_train: 16#' train_configs/stage1_lora_ml1m.yaml
!python train_qformer.py --cfg-path train_configs/stage1_lora_ml1m.yaml 2>&1 | tail -30

In [ ]:
import glob, shutil
best = sorted(glob.glob('/content/logs/stage1_ml1m/*/checkpoint_best.pth'))[-1]
shutil.copy(best, '/content/ckpt/stage1_lora_ml1m.pth'); print(best)
!sed -i 's#ckpt_lora: .*#ckpt_lora: /content/ckpt/stage1_lora_ml1m.pth#' \
    train_configs/stage2_qformer_ml1m.yaml train_configs/stage3_qformer_ml1m.yaml

## 8. Stage 2 — train the Q-Former CIE (short prompt)

Watch the `[qformer-diag]` blocks. Healthy: `pref_token_norm` within 2x of
`llm_emb_norm`; `token_cosine_offdiag` < 0.4; per-token `top3` attention types differ;
`rank_pairs_per_batch` >> 0. And `valid_uauc` must be a real number, not NaN.

In [ ]:
!python train_qformer.py --cfg-path train_configs/stage2_qformer_ml1m.yaml 2>&1 | tail -60

In [ ]:
# the diagnostics trail
import glob, json
f = sorted(glob.glob('/content/logs/stage2_ml1m/*/qformer_diagnostics.jsonl'))[-1]
for l in open(f):
    d = json.loads(l); s = d['summary']
    print(f"{d['tag']:>10s} cos={s['token_cosine_offdiag']:+.4f} "
          f"z_norm={s['pref_token_norm']:.4f} (llm={s['llm_emb_norm']:.4f}) "
          f"z_std={s['z_std_mean']:.4f} pairs={s['rank_pairs_per_batch']:.1f} "
          f"| hist={s['history_source']}/{s['k_hist']} "
          f"filled={s['hist_slots_filled']:.1f} unk={s['hist_unk_rate']:.2%}")
# hist_slots_filled reads ~7 on TRAIN batches (the grouped sampler draws users
# uniformly, and early rows have short histories) and ~33-36 on the eval splits.

In [ ]:
# validation trail: confirm UAUC is finite and improving
import glob, json
log = sorted(glob.glob('/content/logs/stage2_ml1m/*/log.txt'))[-1]
for l in open(log):
    l = l.strip()
    if l.startswith('{') and 'valid_uauc' in l:
        d = json.loads(l)
        print(f"uauc={d['valid_uauc']:.4f} auc={d['valid_auc']:.4f} "
              f"best_epoch={d['valid_best_epoch']} tokens={d['valid_mean_prompt_tokens']} "
              f"users={d['valid_uauc_users_scored']}/+{d['valid_uauc_users_skipped']} skipped")

## 9. Stage 3 — joint tuning, LoRA at lr/10

Mirrors CoLLM's T1/T2 tuning — where its best numbers came from. Needs **both**
checkpoints (stage-2 saves only what was trainable in stage 2). Verify
`n_unexpected=0` on both loads.

In [ ]:
import glob
best2 = sorted(glob.glob('/content/logs/stage2_ml1m/*/checkpoint_best.pth'))[-1]
!sed -i "s#^  ckpt: .*#  ckpt: $best2#" train_configs/stage3_qformer_ml1m.yaml
!grep -E "^  ckpt" train_configs/stage3_qformer_ml1m.yaml
!python train_qformer.py --cfg-path train_configs/stage3_qformer_ml1m.yaml 2>&1 | tail -40

## 10. Evaluation — overall / warm / cold, with prompt-length and timing

In [ ]:
import glob
best3 = sorted(glob.glob('/content/logs/stage3_ml1m/*/checkpoint_best.pth'))[-1]
!python train_qformer.py --cfg-path train_configs/stage3_qformer_ml1m.yaml \
    --options run.evaluate=True model.ckpt=$best3 \
              "run.test_splits=[test,test_warm,test_cold,valid]" \
              run.output_dir=/content/logs/eval_ml1m 2>&1 | grep -E "auc:|Total time" 

## 11. The type_bias heatmap (deliverable 6)

`L x T`: which preference token learned to read which memory slot type.

In [ ]:
import json, glob, numpy as np, matplotlib.pyplot as plt
f = sorted(glob.glob('/content/logs/stage3_ml1m/*/qformer_diagnostics.jsonl'))[-1]
d = [json.loads(l) for l in open(f)][-1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, key, title in zip(axes, ["type_bias_softmax", "attn_by_type"],
                          ["learned type_bias (softmax)", "actual attention mass"]):
    m = np.array(d[key]); im = ax.imshow(m, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(d["slot_type_names"]))); ax.set_xticklabels(d["slot_type_names"])
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels([f"token{i}" for i in range(m.shape[0])])
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig('/content/type_bias_heatmap.png', dpi=150); plt.show()

## 12. Search and ablations

Search on **ML-1M only**, `freeze_rec: True`, one seed, in this order — stop when the
target is met. Then 3 seeds for the final ML-1M row only. Amazon-Book gets the winning
config only. Full command list: runsheet 6.1 / 6.3.

In [ ]:
# example: the L sweep
for L in [2, 4, 8]:
    !python train_qformer.py --cfg-path train_configs/stage2_qformer_ml1m.yaml \
        --options model.qformer.n_query=$L run.output_dir=/content/logs/abl_L$L 2>&1 | tail -3

# k_hist needs NO index rebuild now -- history is per-row. Capped by pit_hist_width (50).
for K in [10, 20, 50]:
    !python train_qformer.py --cfg-path train_configs/stage2_qformer_ml1m.yaml \
        --options model.qformer.memory.k_hist=$K \
                  run.output_dir=/content/logs/abl_khist$K 2>&1 | tail -3

In [ ]:
# the row reviewers will look for: -anticollapse (report the metric drop AND token cosine)
!python train_qformer.py --cfg-path train_configs/stage2_qformer_ml1m.yaml \
    --options model.loss.lambda_div=0 model.loss.lambda_attn=0 model.loss.lambda_var=0 \
              run.output_dir=/content/logs/abl_anticollapse 2>&1 | tail -5

### The `-pit-history` ablation

Reverts the history slots to the old per-user, train-split-only lookup. This is the row
that quantifies what the point-in-time history is worth; everything else is unchanged.

In [ ]:
!python train_qformer.py --cfg-path train_configs/stage2_qformer_ml1m.yaml \
    --options model.qformer.memory.history_source=train_only \
              model.qformer.memory.k_hist=10 \
              run.output_dir=/content/logs/abl_pit_history 2>&1 | tail -5